In [ ]:
deps_path = '/kaggle/input/datasets/nhhsag12/colpali-dependency'
!pip install --no-index --find-links {deps_path} --requirement {deps_path}/requirements.txt

In [ ]:
# --- PATCH: Fix colpali_engine import conflict ---
import types, sys

def _patch_colpali_stub_all():
    problem_modules = [
        "colpali_engine.models.gemma3",
        "colpali_engine.models.gemma3.bigemma3",
        "colpali_engine.models.gemma3.colgemma3",
        "colpali_engine.models.modernvbert",
        "colpali_engine.models.modernvbert.bivbert",
        "colpali_engine.models.modernvbert.colvbert",
        "colpali_engine.models.paligemma",
        "colpali_engine.models.paligemma.bipali",
        "colpali_engine.models.paligemma.colpali",
        "colpali_engine.models.paligemma.bipali_proj",
        "colpali_engine.models.qwen2",
        "colpali_engine.models.qwen2.biqwen2",
        "colpali_engine.models.qwen2.colqwen2",
        "colpali_engine.models.qwen3",
        "colpali_engine.models.qwen3.biqwen3",
        "colpali_engine.models.qwen3.colqwen3",
        "colpali_engine.models.qwen3_5",
        "colpali_engine.models.qwen3_5.biqwen3_5",
        "colpali_engine.models.qwen3_5.colqwen3_5",
        "colpali_engine.models.qwen_omni",
        "colpali_engine.models.qwen_omni.colqwen2_5_omni",
    ]

    stub_class_map = {
        "colpali_engine.models.gemma3": [
            "BiGemma3", "BiGemmaProcessor3", "ColGemma3", "ColGemmaProcessor3",
        ],
        "colpali_engine.models.modernvbert": [
            "BiModernVBert", "BiModernVBertProcessor", "ColModernVBert", "ColModernVBertProcessor",
        ],
        "colpali_engine.models.paligemma": [
            "BiPali", "BiPaliProcessor", "BiPaliProj", "ColPali", "ColPaliProcessor",
        ],
        "colpali_engine.models.qwen2": [
            "BiQwen2", "BiQwen2Processor", "ColQwen2", "ColQwen2Processor",
        ],
        "colpali_engine.models.qwen3": [
            "BiQwen3", "BiQwen3Processor", "ColQwen3", "ColQwen3Processor",
        ],
        "colpali_engine.models.qwen3_5": [
            "BiQwen3_5", "BiQwen3_5Processor", "ColQwen3_5", "ColQwen3_5Processor",
        ],
        "colpali_engine.models.qwen_omni": [
            "ColQwen2_5Omni", "ColQwen2_5OmniProcessor",
        ],
    }

    for name in problem_modules:
        stub = types.ModuleType(name)
        sys.modules[name] = stub

    for mod_name, cls_list in stub_class_map.items():
        mod = sys.modules[mod_name]
        for cls_name in cls_list:
            setattr(mod, cls_name, type(cls_name, (), {}))

    stale = [k for k in sys.modules if k.startswith("colpali_engine") and k not in problem_modules]
    for k in stale:
        del sys.modules[k]

    print(f">>> Stubbed {len(problem_modules)} problematic submodules")

_patch_colpali_stub_all()

from colpali_engine.models.idefics3 import ColIdefics3, ColIdefics3Processor
print(f"✅ ColIdefics3 ready: {ColIdefics3}")

In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

In [ ]:
# --- BƯỚC 1: SETUP DATA ---
import glob
import os
import pandas as pd
import numpy as np
import json
from tqdm.notebook import tqdm

# ==============================================================================
# CONFIG
# ==============================================================================
COLSMOL_DIR = "/kaggle/input/datasets/nguyenducdung1107/colsmol500m-layoutmmdoc/colsmol500m-pkl"
SEARCH_ROOT = "/kaggle/input"
ANNOTATIONS_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
PARQUET_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_layouts.parquet"

CURRENT_BATCH_IDX = 0 # chỉnh batch ở đây

# ==============================================================================
# STEP 1: LOAD PKL FILES & BUILD BATCH RANGES (ROBUST)
# ==============================================================================
pkl_files = sorted(glob.glob(os.path.join(COLSMOL_DIR, "*.pkl")))

print(f"Found {len(pkl_files)} PKL files")

if len(pkl_files) == 0:
    raise ValueError("❌ No PKL files found. Check COLSMOL_DIR path!")

BATCH_RANGES = []

for pkl_file in pkl_files:
    base = os.path.basename(pkl_file).replace('.pkl', '')
    
    if '-' in base:
        try:
            start, end = map(int, base.split('-'))
            BATCH_RANGES.append((start, end))
        except:
            pass

# fallback nếu parse fail
if len(BATCH_RANGES) == 0:
    print("⚠️ Using fallback batch indexing")
    BATCH_RANGES = [(i, i+1) for i in range(len(pkl_files))]

BATCH_RANGES = sorted(BATCH_RANGES)

print(f"Found {len(BATCH_RANGES)} batch ranges:")
for start, end in BATCH_RANGES[:5]:
    print(f"  [{start}:{end}]")

# validate index
if CURRENT_BATCH_IDX >= len(BATCH_RANGES):
    raise IndexError(f"❌ CURRENT_BATCH_IDX={CURRENT_BATCH_IDX} out of range")

START_IDX, END_IDX = BATCH_RANGES[CURRENT_BATCH_IDX]
print(f"\n>>> Processing batch [{START_IDX}:{END_IDX}]")

# ==============================================================================
# STEP 2: AUTO DETECT PATHS
# ==============================================================================
ENHANCED_JSONL_DIR = None
ENHANCED_IMG_DIR = None

for root, dirs, files in os.walk(SEARCH_ROOT):
    if "LAYOUT_CONTENT_FINAL" in root:
        ENHANCED_JSONL_DIR = root
    if "IMAGE ENHACED" in root or (sum(1 for f in files if f.endswith(".jpg")) > 1000):
        ENHANCED_IMG_DIR = root

# fallback
if not ENHANCED_JSONL_DIR:
    ENHANCED_JSONL_DIR = "/kaggle/input/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/LAYOUT_CONTENT_FINAL"

if not ENHANCED_IMG_DIR:
    ENHANCED_IMG_DIR = "/kaggle/input/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/IMAGE ENHACED"

print(f"Enhanced JSONL DIR: {ENHANCED_JSONL_DIR}")
print(f"Enhanced IMG DIR: {ENHANCED_IMG_DIR}")

# ==============================================================================
# STEP 3: BUILD DOC MAPPING (NO GLOBAL BUG)
# ==============================================================================
print("Building document mapping...")

valid_docs_in_qa = set()
with open(ANNOTATIONS_PATH, 'r') as f:
    for line in f:
        try:
            d = json.loads(line)
            valid_docs_in_qa.add(d['doc_name'].replace('.pdf', ''))
        except:
            pass

available_jsonls = glob.glob(os.path.join(ENHANCED_JSONL_DIR, "*.jsonl"))

jsonl_map = {}
for p in available_jsonls:
    fname = os.path.basename(p).replace('_layout.jsonl', '')
    jsonl_map[fname] = p

intersection_docs = sorted(list(valid_docs_in_qa.intersection(jsonl_map.keys())))
print(f"→ Found {len(intersection_docs)} valid documents")

# ==============================================================================
# STEP 4: SELECT BATCH DOCS
# ==============================================================================
target_doc_names = intersection_docs[START_IDX:END_IDX]
target_files = [jsonl_map[d] for d in target_doc_names if d in jsonl_map]

print(f"Processing batch [{START_IDX}:{END_IDX}]")
print(f"→ Docs: {len(target_doc_names)}")

if len(target_doc_names) == 0:
    raise ValueError("❌ Batch is empty! Check batch index or data.")

# ==============================================================================
# STEP 5: LOAD DATA
# ==============================================================================
print("Loading Parquet...")
df_orig = pd.read_parquet(PARQUET_PATH)
df_orig['join_doc_name'] = df_orig['doc_name'].str.replace('.pdf', '', regex=False)
df_orig = df_orig[df_orig['join_doc_name'].isin(target_doc_names)]

print("Loading JSONL enrichment...")
dfs = []

for f in tqdm(target_files, desc="Reading JSONLs"):
    try:
        temp = pd.read_json(f, lines=True)
        temp['join_doc_name'] = os.path.basename(f).replace('_layout.jsonl', '')
        
        if 'layout' in temp.columns:
            temp = temp.rename(columns={'layout': 'layout_id'})
        
        cols = ['join_doc_name', 'layout_id', 'vlm_text', 'img_enhanced_path']
        if 'text_level' in temp.columns:
            cols.append('text_level')
        
        temp = temp[[c for c in cols if c in temp.columns]]
        dfs.append(temp)
    except Exception as e:
        print(f"Skip {f}: {e}")

if len(dfs) > 0:
    df_enh = pd.concat(dfs, ignore_index=True)
    df_enh = df_enh.rename(columns={
        'vlm_text': 'vlm_text_enhanced',
        'text_level': 'text_level_enhanced'
    })
else:
    df_enh = pd.DataFrame()

# ==============================================================================
# STEP 6: MERGE
# ==============================================================================
print("Merging data...")
df_final = pd.merge(df_orig, df_enh, on=['join_doc_name', 'layout_id'], how='left')
df_final = df_final.sort_values(by=['join_doc_name', 'page_id', 'layout_id'])

# ==============================================================================
# STEP 7: CONTEXT BUILDING
# ==============================================================================
def identify_header(row):
    if row.get('type') in ['title', 'section_header', 'header']:
        return str(row.get('text', ''))
    if pd.notna(row.get('text_level_enhanced')):
        return str(row.get('text', ''))
    return np.nan

df_final['temp_header'] = df_final.apply(identify_header, axis=1)
df_final['current_section'] = (
    df_final.groupby('join_doc_name')['temp_header']
    .ffill()
    .fillna("General Content")
)

# ==============================================================================
# STEP 8: IMAGE MAP
# ==============================================================================
enh_image_map = {}

if os.path.exists(ENHANCED_IMG_DIR):
    for f in glob.glob(os.path.join(ENHANCED_IMG_DIR, "*")):
        enh_image_map[os.path.basename(f)] = f

# ==============================================================================
# STEP 9: FINAL SOURCE SELECTION
# ==============================================================================
def get_best_sources(row):
    img_type, img_data = None, None

    # image priority
    if pd.notna(row.get('img_enhanced_path')):
        fname = os.path.basename(str(row['img_enhanced_path']))
        if fname in enh_image_map:
            img_type, img_data = 'path', enh_image_map[fname]

    if img_data is None and pd.notna(row.get('image_binary')):
        img_type, img_data = 'binary', row['image_binary']

    # text priority
    raw_content = ""

    if pd.notna(row.get('vlm_text_enhanced')) and len(str(row['vlm_text_enhanced'])) > 5:
        raw_content = str(row['vlm_text_enhanced'])
    elif pd.notna(row.get('text')) and len(str(row['text'])) > 5:
        raw_content = str(row['text'])
    elif pd.notna(row.get('ocr_text')) and len(str(row['ocr_text'])) > 5:
        raw_content = str(row['ocr_text'])
    elif pd.notna(row.get('vlm_text')):
        raw_content = str(row['vlm_text'])

    section = row.get('current_section', '')
    final_text_prompt = f"Section: {section}\nContent: {raw_content}"

    if len(final_text_prompt) < 10:
        final_text_prompt = "Document layout."

    return pd.Series(
        [img_type, img_data, final_text_prompt],
        index=['img_type', 'img_data', 'final_text']
    )

print("Building final dataset...")
processed = df_final.apply(get_best_sources, axis=1)

sample_layouts_df = (
    pd.concat([df_final, processed], axis=1)
    .dropna(subset=['img_type'])
    .reset_index(drop=True)
)

# ==============================================================================
# DONE
# ==============================================================================
print("=" * 60)
print(f"✅ BATCH [{START_IDX}-{END_IDX}] READY!")
print(f"Total Layouts: {len(sample_layouts_df)}")
print("=" * 60)

In [ ]:
# --- BƯỚC 2: LOAD MODEL + LORA ---
print(">>> Loading ColSmolVLM-500M + LoRA...")

import torch
import gc
from peft import PeftModel
from colpali_engine.models import ColIdefics3, ColIdefics3Processor

gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"

BASE_MODEL = "/kaggle/input/datasets/nguyenducdung1107/model500m"
LORA_PATH = "/kaggle/input/datasets/nguyenducdung1107/colsmol500-adapter/colsmol500_adapter"  # <-- đường dẫn adapter của m

# 🔹 Load base model
model = ColIdefics3.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager"
)

# 🔹 Load LoRA adapter
model = PeftModel.from_pretrained(
    model,
    LORA_PATH
)

model.eval()

# 🔹 Processor
processor = ColIdefics3Processor.from_pretrained(BASE_MODEL)

print(" Model + LoRA Ready!")

In [ ]:
query_model=model
query_processor=processor

In [ ]:
# ==============================================================================
# CELL 1 — INFRASTRUCTURE  (chạy 1 lần, không đổi dù method thay đổi)
#
# Bao gồm:
#   - Imports & config TĨNH (paths, batch sizes, data configs)
#   - Trackers: TFLOPs, WallClock, Memory
#   - CUDA timing util
#   - Pure scoring utils: uniform_maxsim_scores, topk_query_tokens, flops_*
#   - Query encoding: build_content_mask, encode_all_queries
#   - Doc matrix builder: build_full
#   - Metric helpers: hit_metrics, ndcg, recall_area, ...
#   - Metric store: all_metrics, record(), ...
#   - Data loading: PKL → layouts_df → QA pairs → bbox_list → doc_group_ids
#     (per batch — Cell 2 iterate qua BATCH_RANGES)
#
# KHÔNG bao gồm:
#   - SphericalRVQ / QuantDocStore
#   - QuantileAdaptivePruner (Stage 2)
#   - CertificatePruner (Stage 3)
#   - MSTARReranker (Stage 4)
#   - Config của method: SRVQ_STAGES_LIST, BETA_LIST, ALPHA_LIST, MSTAR params
#   - Eval loop, wall-clock benchmark, summary/save
# ==============================================================================

print(">>> CELL 1: Infrastructure loading ...")

import torch
import torch.nn.functional as F
import numpy as np
import json, os, pickle, gc, glob, time
import pandas as pd
from tqdm.notebook import tqdm

# ==============================================================================
# CONFIG TĨNH — paths, data, batch sizes (không phải method knob)
# ==============================================================================
QUERY_BATCH_SIZE    = 50
ADC_DOC_CHUNK       = 4000
WALLCLOCK_N_WARMUP  = 5
WALLCLOCK_N_MEASURE = 50
TOPK_RATIOS         = [0.3, 0.5, 0.7, 1.0]   # query-side ablation baselines

WORKING_DIR        = "/kaggle/working"
ANNOTATIONS_PATH   = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
COLSMOL_DIR        = "/kaggle/input/datasets/nguyenducdung1107/colsmol500m-layoutmmdoc/colsmol500m-pkl"
ENHANCED_JSONL_DIR = "/kaggle/input/datasets/cdnghnam/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/LAYOUT_CONTENT_FINAL"
ENHANCED_IMG_DIR   = "/kaggle/input/datasets/cdnghnam/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/IMAGE ENHACED"
PARQUET_PATH       = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_layouts.parquet"
BATCH_RANGE_PKL_OVERRIDE = {
    (0, 25): "/kaggle/input/datasets/nguyenducdung1107/colsmol500m-layoutmmdoc/colsmol500m-pkl/0-25.pkl",
}

device = "cuda" if torch.cuda.is_available() else "cpu"

# ==============================================================================
# MODEL ACCESSOR
# ==============================================================================
def get_model_and_processor():
    missing = [v for v in ('query_model', 'query_processor') if v not in globals()]
    if missing:
        raise RuntimeError(f"Variables {missing} not found. Run model load cell first.")
    return globals()['query_model'], globals()['query_processor']

try:
    _m, _p = get_model_and_processor()
    print(f"✅ Model ready: {type(_m).__name__}")
    del _m, _p
except RuntimeError as _e:
    print(f"❌ {_e}"); raise

# ==============================================================================
# TRACKERS
# ==============================================================================
class TFLOPsTracker:
    def __init__(self): self._d = {}
    def add(self, key, subkey, flops):
        self._d.setdefault(key, {}).setdefault(subkey, []).append(flops / 1e12)
    def mean(self, key, subkey='default'):
        v = self._d.get(key, {}).get(subkey, [])
        return float(np.mean(v)) if v else float('nan')
    def to_df(self):
        rows = []
        for m, rd in self._d.items():
            for sk, v in rd.items():
                rows.append({'method': m, 'subkey': sk,
                             'avg_tflops': round(float(np.mean(v)), 8), 'n': len(v)})
        return pd.DataFrame(rows).sort_values(['method', 'subkey']).reset_index(drop=True)


class WallClockTracker:
    def __init__(self): self._times = {}
    def record(self, key, ms):
        self._times.setdefault(key, []).append(ms)
    def stats(self, key):
        v = self._times.get(key, [])
        if not v:
            return {'median_ms': float('nan'), 'mean_ms': float('nan'),
                    'std_ms': float('nan'), 'n': 0}
        return {'median_ms': float(np.median(v)), 'mean_ms': float(np.mean(v)),
                'std_ms': float(np.std(v)), 'n': len(v)}
    def to_df(self):
        rows = []
        for k, v in self._times.items():
            if not v: continue
            rows.append({'method': k,
                         'median_ms': round(float(np.median(v)), 4),
                         'mean_ms':   round(float(np.mean(v)),   4),
                         'std_ms':    round(float(np.std(v)),    4),
                         'n':         len(v)})
        if not rows:
            return pd.DataFrame(columns=['method', 'median_ms', 'mean_ms', 'std_ms', 'n'])
        return pd.DataFrame(rows).sort_values('median_ms').reset_index(drop=True)


class MemoryTracker:
    def __init__(self): self._entries = {}
    def record_tensor(self, key, *tensors):
        total = sum(t.numel() * t.element_size()
                    for t in tensors if isinstance(t, torch.Tensor))
        self._entries[key] = total / (1024 ** 2)
    def record_mb(self, key, mb): self._entries[key] = mb
    def get(self, key): return self._entries.get(key, float('nan'))
    def to_df(self):
        return pd.DataFrame([{'method': k, 'memory_mb': round(v, 2)}
                              for k, v in self._entries.items()]
                             ).sort_values('memory_mb').reset_index(drop=True)


tflops_tracker = TFLOPsTracker()
wallclock      = WallClockTracker()
mem_tracker    = MemoryTracker()

# ==============================================================================
# CUDA TIMING UTIL
# ==============================================================================
def cuda_time_ms(fn, device, n_warmup=0):
    for _ in range(n_warmup): fn()
    is_cuda = (str(device) in ('cuda', 'cuda:0') or
               (isinstance(device, torch.device) and device.type == 'cuda'))
    if is_cuda:
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        result = fn()
        torch.cuda.synchronize()
    else:
        t0 = time.perf_counter()
        result = fn()
    return (time.perf_counter() - t0) * 1000.0, result

# ==============================================================================
# FLOP COUNTERS
# ==============================================================================
def flops_maxsim(Q, N, L, D): return 2.0 * Q * N * L * D
def flops_adc_rvq(Q, N, L, S, K, D): return float(S * (2.0 * Q * K * D + Q * N * L))

# ==============================================================================
# SCORING UTILS  — không phụ thuộc method
# ==============================================================================
def uniform_maxsim_scores(q_norm, doc_matrix, doc_mask, chunk_size=ADC_DOC_CHUNK):
    """Standard MaxSim: sum over query tokens of max-over-doc-tokens similarity."""
    Q   = q_norm.shape[0]
    N   = doc_matrix.shape[0]
    dev = q_norm.device
    out = torch.zeros(N, device=dev)
    for s in range(0, N, chunk_size):
        e   = min(s + chunk_size, N)
        sim = torch.einsum('qd,nld->qnl', q_norm.float(), doc_matrix[s:e].float())
        sim.masked_fill_(~doc_mask[s:e].unsqueeze(0), float('-inf'))
        ms  = sim.max(dim=-1).values
        ms  = ms.masked_fill(ms == float('-inf'), 0.0)
        out[s:e] = ms.sum(0)
    return out


def topk_query_tokens(q_norm, magnitude, ratio):
    """Query-side top-k pruning baseline — selects top-ratio fraction by magnitude."""
    Q = q_norm.shape[0]
    k = max(1, int(Q * ratio))
    if k >= Q:
        return q_norm, torch.arange(Q, device=q_norm.device)
    topk_idx = magnitude.topk(k).indices.sort().values
    return q_norm[topk_idx], topk_idx

# ==============================================================================
# [P1] QUERY ENCODING
# ==============================================================================
def build_content_mask(inputs, processor):
    """Mask out special tokens (pad/bos/eos/...) — tokenizer-only logic."""
    attn_mask = inputs["attention_mask"]
    input_ids = inputs.get("input_ids", None)
    if input_ids is None:
        return attn_mask.float()
    tok = getattr(processor, 'tokenizer', processor)
    special_ids = set()
    for attr in ['pad_token_id', 'bos_token_id', 'eos_token_id',
                 'unk_token_id', 'sep_token_id', 'cls_token_id']:
        tid = getattr(tok, attr, None)
        if tid is not None: special_ids.add(int(tid))
    if hasattr(tok, 'added_tokens_encoder'):
        for _, tid in tok.added_tokens_encoder.items():
            special_ids.add(int(tid))
    if not special_ids:
        return attn_mask.float()
    special_tensor = torch.tensor(list(special_ids), device=input_ids.device)
    is_special = (input_ids.unsqueeze(-1) == special_tensor).any(dim=-1)
    return attn_mask.float() * (~is_special).float()


def encode_all_queries(qa_pairs, processor, model, device):
    """
    Encode all queries to normalized token embeddings + magnitude.
    Returns list of dicts: q_norm, magnitude, Mc, gt_set, lmr, domain, doc_name, question, gq
    """
    encoded = []
    for i in range(0, len(qa_pairs), QUERY_BATCH_SIZE):
        batch = qa_pairs[i:i + QUERY_BATCH_SIZE]
        q_in  = processor.process_queries([it['question'] for it in batch]).to(device)
        with torch.no_grad():
            q_out = model(**{k: v for k, v in q_in.items()})
            q_embs = q_out.float() if isinstance(q_out, torch.Tensor) \
                     else q_out.last_hidden_state.float()
        cmasks = build_content_mask(q_in, processor).float()
        for j, item in enumerate(batch):
            cidx        = torch.where(cmasks[j] > 0)[0]
            raw_content = q_embs[j][cidx]
            magnitude   = raw_content.norm(dim=-1)
            q_norm      = F.normalize(raw_content, dim=-1)
            encoded.append({
                'q_norm':    q_norm.cpu(),
                'magnitude': magnitude.cpu(),
                'Mc':        cidx.numel(),
                'gt_set':    item['gt_pos_set'],
                'lmr':       item['layout_mapping_raw'],
                'domain':    item['domain'],
                'doc_name':  item['doc_name'],
                'question':  item['question'],
                'gq':        i + j,
            })
        del q_in, q_embs, cmasks
    return encoded

# ==============================================================================
# [P2] DOC MATRIX BUILDER
# ==============================================================================
def _coerce(item):
    if isinstance(item, torch.Tensor): return item
    if isinstance(item, dict):
        for k in ('embedding', 'embeddings', 'vector', 'vectors', 'token_embeddings'):
            if k in item and isinstance(item[k], torch.Tensor): return item[k]
        tvs = [(k, v) for k, v in item.items() if isinstance(v, torch.Tensor)]
        if tvs: return tvs[0][1]
    raise ValueError(f"Cannot coerce type: {type(item)}")


def build_full(docs_list, device):
    """Build padded + normalized doc matrix (float16 on CUDA). Unchanged across methods."""
    ts = [_coerce(x) for x in docs_list]
    ts = [t.squeeze(0) if t.dim() > 2 else (t.unsqueeze(0) if t.dim() == 1 else t)
          for t in ts]
    N    = len(ts)
    Lmax = max(t.shape[0] for t in ts)
    D    = ts[0].shape[1]
    dtype = torch.float16 if torch.device(device).type == 'cuda' else torch.float32
    pad  = torch.zeros(N, Lmax, D, device=device, dtype=dtype)
    mask = torch.zeros(N, Lmax, device=device, dtype=torch.bool)
    for i, t in enumerate(ts):
        L = t.shape[0]
        pad[i, :L]  = F.normalize(t.float().to(device), dim=-1).to(dtype)
        mask[i, :L] = True
    return pad, mask

# ==============================================================================
# METRIC HELPERS
# ==============================================================================
def _parse_bbox(raw):
    if raw is None: return None
    try:
        if isinstance(raw, np.ndarray):
            f = raw.flatten()
            return [float(x) for x in f] if len(f) == 4 else None
    except: pass
    if isinstance(raw, (list, tuple)) and len(raw) == 4:
        try: return [float(x) for x in raw]
        except: return None
    if isinstance(raw, dict):
        for keys in [('x1','y1','x2','y2'), ('top','left','bottom','right'),
                     ('left','top','right','bottom')]:
            if all(k in raw for k in keys):
                try: return [float(raw[k]) for k in keys]
                except: pass
    return None


def overlap_area(b1, b2):
    it = max(b1[0], b2[0]); il = max(b1[1], b2[1])
    ib = min(b1[2], b2[2]); ir = min(b1[3], b2[3])
    return (ib - it) * (ir - il) if it < ib and il < ir else 0.0


def recall_area(top_k, bbox_list, lmr):
    ra = 0.0
    for p in top_k:
        info = bbox_list[p] if 0 <= p < len(bbox_list) else None
        if info is None: continue
        pg, t, l, b, r = info
        for gt in lmr:
            if pg != gt["page"]: continue
            gb = _parse_bbox(gt.get("bbox"))
            if gb: ra += overlap_area([t, l, b, r], gb)
    ga = 0.0
    for gt in lmr:
        gb = _parse_bbox(gt.get("bbox"))
        if gb: t2, l2, b2, r2 = gb; ga += max(0, b2-t2) * max(0, r2-l2)
    return 0.0 if ga <= 0 else min(ra / ga, 1.0)


def ndcg(ranked, gt, k):
    dcg  = sum(1 / np.log2(r + 2) for r, i in enumerate(ranked[:k]) if i in gt)
    idcg = sum(1 / np.log2(r + 2) for r in range(min(len(gt), k)))
    return dcg / idcg if idcg > 0 else 0.0


def hit_metrics(top10, gt_set, bbox_list, lmr):
    if not lmr: return None
    h = next((r + 1 for r, i in enumerate(top10) if i in gt_set), -1)
    return {
        'r1':       int(h != -1 and h <= 1),
        'r5':       int(h != -1 and h <= 5),
        'r10':      int(h != -1 and h <= 10),
        'recall1':  recall_area(top10[:1],  bbox_list, lmr),
        'recall5':  recall_area(top10[:5],  bbox_list, lmr),
        'recall10': recall_area(top10[:10], bbox_list, lmr),
        'n1':       ndcg(top10, gt_set, 1),
        'n5':       ndcg(top10, gt_set, 5),
        'n10':      ndcg(top10, gt_set, 10),
    }

# ==============================================================================
# METRIC STORE
# ==============================================================================
def _init_m():
    return {'r1': 0, 'r5': 0, 'r10': 0, 'n1': 0., 'n5': 0., 'n10': 0.,
            'recall1': 0., 'recall5': 0., 'recall10': 0., 'count': 0}

def _add(d, s):
    for f in ('r1', 'r5', 'r10'): d[f] += int(s[f])
    for f in ('n1', 'n5', 'n10', 'recall1', 'recall5', 'recall10'): d[f] += float(s[f])
    d['count'] += 1

def _ens(store, k):
    if k not in store: store[k] = _init_m()
    return store[k]

all_metrics        = {}
all_domain_metrics = {}
all_query_results  = []
all_batch_stats    = []

def record(key, m, domain):
    _add(_ens(all_metrics,  key), m)
    if domain not in all_domain_metrics: all_domain_metrics[domain] = {}
    _add(_ens(all_domain_metrics[domain], key), m)

# ==============================================================================
# LOAD INDEX FILES & BUILD BATCH LIST
# ==============================================================================
pkl_files = sorted(glob.glob(os.path.join(COLSMOL_DIR, "*.pkl")))
print(f"Found {len(pkl_files)} PKL files")

BATCH_RANGES = []
for p in pkl_files:
    base = os.path.basename(p).replace('.pkl', '')
    try:
        s, e = map(int, base.split(' ')[-1].split('-'))
        BATCH_RANGES.append((s, e))
    except: pass
BATCH_RANGES = sorted(BATCH_RANGES) or [(i, i+1) for i in range(len(pkl_files))]
for r in list(BATCH_RANGE_PKL_OVERRIDE.keys()):
    if r not in BATCH_RANGES:
        BATCH_RANGES = sorted(set(BATCH_RANGES + [r]))

valid_docs = set()
with open(ANNOTATIONS_PATH) as f:
    for line in f:
        try: valid_docs.add(json.loads(line)['doc_name'].replace('.pdf', ''))
        except: pass

jsonl_map = {
    os.path.basename(p).replace('_layout.jsonl', ''): p
    for p in glob.glob(os.path.join(ENHANCED_JSONL_DIR, "*.jsonl"))
}
intersection_docs = sorted(valid_docs.intersection(jsonl_map.keys()))
print(f"Intersection docs: {len(intersection_docs)}")
print(f"Batch ranges     : {BATCH_RANGES}")
print("\n>>> CELL 1 DONE — run Cell 2 to train quantizers and evaluate")